# 04 - Feature Engineering

**Goal for today:** turn our raw, mostly-text dataset into a fully numeric table a
model can actually learn from. By the end, every column will be a number, there will
be zero missing values, and we'll save the result so notebook 05 can load it directly.

**Why this matters for MLA-C01 (Domain 1 - Data Preparation for ML):**
This is the heart of Domain 1. The exam expects you to know *why* certain
transformations are needed (models are just math - they cannot directly interpret the
string `"Month-to-month"`), and to know the standard techniques: imputation for
missing values, and encoding for categorical data. On AWS, this is exactly what a
**SageMaker Processing job** or **Data Wrangler transform** would do before writing
data out for training.

## Step 0: Reload the data

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/telco_churn.csv')
df.shape

(7043, 21)

## Step 1: Fix `TotalCharges` dtype (from notebook 02)

In [2]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].isnull().sum()

np.int64(11)

## Step 2: Handle the 11 missing `TotalCharges` values - impute, don't drop

**Why not just drop these 11 rows?** With only 7,043 total rows, throwing away data
should always be a deliberate, last-resort decision - every row is a real customer's
information. The general term for filling in missing values instead of removing them
is **imputation**, and the right imputation strategy depends on *why* the value is
missing - which is exactly why we investigated this back in notebook 02.

Recall what we found: all 11 rows have `tenure == 0` - these are brand new customers
who haven't been billed yet. That's not random noise, it's a logical fact: **a
customer with zero tenure has logically been charged $0 in total.** So instead of a
generic strategy like filling with the column average, we can use domain knowledge to
fill these with `0` - the value that's actually correct here, not just a statistical
placeholder.

In [3]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['TotalCharges'].isnull().sum()

np.int64(0)

**A quick note on why this matters for the exam:** if we had instead filled these
with the column *mean* or *median*, we would have quietly invented total-charge
history for customers who never had any - a subtle way to introduce bias into your
data. Domain 1 tests whether you understand that imputation strategy should be chosen
based on the actual mechanism behind the missingness, not applied blindly.

## Step 3: Drop `customerID`

**What this cell does:** `.drop(columns=[...])` removes one or more columns.

**Why drop it:** `customerID` is a unique identifier - a different value for every
single row, with zero relationship to whether someone churns. Including identifier
columns in training data is a classic mistake: some models can accidentally treat a
high-cardinality ID-like column as informative (a form of overfitting to noise), and
even when they don't, it adds nothing but computational cost. Always ask
"could this column contain any real signal, or is it just a label?" before including
a feature.

In [4]:
df = df.drop(columns=['customerID'])
df.shape

(7043, 20)

## Step 4: Encode the target column (`Churn`) as 0/1

**What this cell does:** `.map({...})` replaces each value in a column according to
a dictionary you provide - here, `'Yes'` becomes `1` and `'No'` becomes `0`.

**Why:** the target column needs to be numeric for virtually every ML algorithm to
train on it. By convention, `1` represents the "positive class" (the thing we're
trying to detect - churn) and `0` the "negative class." This convention matters later
when we discuss precision/recall in notebook 05.

In [5]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['Churn'].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

## Step 5: One-hot encode the remaining categorical columns

**What this cell does:** `pd.get_dummies()` is pandas' built-in **one-hot encoding**
function. For every categorical column, it creates a new binary (0/1) column for each
possible category, and marks a `1` in whichever one applies to that row - all others
get `0`.

Example: a `Contract` column with values `Month-to-month` / `One year` / `Two year`
becomes three new columns - `Contract_Month-to-month`, `Contract_One year`,
`Contract_Two year` - each holding `True`/`False` (which behaves as 1/0).

**Why not just assign numbers directly** (e.g. Month-to-month=0, One year=1,
Two year=2)? Because that would imply a false *order* and *distance* between
categories - the model might learn "Two year is twice as much as One year," which is
meaningless for a category with no natural ranking. One-hot encoding avoids that by
treating each category as its own independent yes/no feature.

`drop_first=True` drops one category per column to avoid redundancy - if you know a
customer is *not* Month-to-month and *not* One year, you already know they must be
Two year, so keeping all three would be duplicating information the model doesn't need.

In [7]:
categorical_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
categorical_cols

['gender',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod']

In [10]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
df_encoded.shape

(7043, 31)

Notice the column count jumped significantly - that's expected. Each categorical
column with N categories becomes (N-1) new columns after `drop_first=True`. More
columns isn't automatically better, but it's the necessary trade-off for representing
categories numerically without inventing a false order.

## Step 6: Final check before saving

In [11]:
df_encoded.dtypes.value_counts()

bool       26
int64       3
float64     2
Name: count, dtype: int64

In [12]:
df_encoded.isnull().sum().sum()

np.int64(0)

Two things to confirm here:
- `df_encoded.dtypes.value_counts()` should show only numeric types (`bool`, `int64`,
  `float64`) - no more `object`/text columns anywhere
- `df_encoded.isnull().sum().sum()` should be `0` - the double `.sum()` first totals
  missing values per column, then adds those totals together into one single number,
  confirming zero missing values across the *entire* dataset

If both of those check out, this DataFrame is genuinely ready for a model.

## Step 7: Save the processed data for notebook 05

**What this cell does:** `.to_csv()` writes the DataFrame to a new CSV file.
`index=False` prevents pandas from adding an extra unnamed column for the row index -
we don't need that saved, since it's not real data.

We're saving this as a **separate file** from the raw `telco_churn.csv`, rather than
overwriting it. That's a deliberate practice: always keep your raw data untouched and
treat cleaned/derived data as a separate, regenerable artifact. If you ever discover a
mistake in this feature engineering step, you can fix the code and regenerate this
file - but if you'd overwritten the raw file, that original would be gone for good.

In [13]:
df_encoded.to_csv('../data/telco_churn_processed.csv', index=False)
print("Saved:", df_encoded.shape)

Saved: (7043, 31)


---

**That's it for today.** Small, complete increment:
- imputed the 11 missing `TotalCharges` values with `0`, using domain knowledge
  (tenure == 0) rather than a generic statistical fill
- dropped `customerID` (pure identifier, no predictive signal)
- encoded `Churn` to 0/1
- one-hot encoded every remaining categorical column with `pd.get_dummies()`
- verified the result is fully numeric with zero missing values
- saved the result to `data/telco_churn_processed.csv`

**Next session:** notebook `05_first_model.ipynb` - load this processed file, split it
into training and test sets, train a first classifier, and properly evaluate it
(accuracy alone won't cut it, given the class imbalance we found back in notebook 01).